# UIT DSC 2026 — v0.5007 clean-adapter A/B

Thí nghiệm A1: giữ chính xác retrieval, dense index, selector, prompt và generation của run public METEOR 0.5007. Chỉ thay full QLoRA adapter bằng adapter clean có fingerprint đã khóa.

Phiên đầu chạy mới với `AUTO_RESUME=False`. Nếu dừng an toàn, chỉ attach recovery do chính A1 tạo ra, rồi đổi `AUTO_RESUME=True`.

In [ ]:
DEVICE = 'cuda:0'

# A1 guard values. Không thay các giá trị này.
EXPECTED_ADAPTER_FINGERPRINT = '590a732d4ea601e20b68d27407206265a52fd5753402ae2f9dc3e7084ccd850a'
EXPECTED_RUN_FINGERPRINT = '8fc470f863a34338'
EXPECTED_DENSE_IDENTITY = 'b5c6eadb46ee7aceab44b8643fa8a5beefde5420c02cf2146e0fd30e38eccf23'
EXPECTED_SELECTOR_SHA256 = '85a00e46e988b013741ab94c0b5bcf4777b6909aff4d2b3b60d00664f06ada38'

# Actual v0.5007 runtime configuration.
EMBEDDING_BATCH_SIZE = 2
RERANKER_BATCH_SIZE = 1
VERIFIER_BATCH_SIZE = 4
RERANKER_MAX_LENGTH = 512
MAX_CONTEXT_CHARS = 18000
MAX_NEW_TOKENS = 900
BM25_TOP_K = 140
DENSE_TOP_K = 70
FUSED_TOP_K = 48
RERANK_TOP_K = 24
EVIDENCE_TOP_K = 5
TRAIN_MEMORY = False
REUSE_RETRIEVAL = True
MODEL_SMOKE = True
FORCE_INDEX = False
FORCE_DENSE_INDEX = False
DENSE_INDEX_ONLY = False
DENSE_INDEX_DIR = '/kaggle/working/legalqa_dense_index'
CHECKPOINT_EVERY = 50
AUTO_RESUME = False  # First session False. Only change to True with A1 recovery attached.
RECOVERY_OUTPUT = '/kaggle/working/legalqa_recovery_latest.zip'
MAX_RUNTIME_HOURS = 11.5

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
input_root = Path('/kaggle/input')
def require_kaggle_cuda():
    nvidia_smi = shutil.which('nvidia-smi')
    if nvidia_smi is None:
        raise RuntimeError('Session Kaggle chưa được gắn GPU. Vào Settings/Session options → Accelerator → GPU (T4 x2 hoặc T4 x1), restart session rồi Run All.')
    import torch
    if not torch.cuda.is_available() or torch.cuda.device_count() < 1:
        raise RuntimeError('Kaggle có nvidia-smi nhưng PyTorch không thấy CUDA. Hãy restart session sau khi bật GPU.')
    subprocess.run([nvidia_smi, '--query-gpu=index,name,memory.total', '--format=csv'], check=True)
    print(f'CUDA OK: {torch.cuda.device_count()} GPU; pipeline dùng cuda:0')
require_kaggle_cuda()
public_files = sorted(input_root.rglob('public-official.json'))
if len(public_files) != 1:
    raise FileNotFoundError(f'Cần duy nhất một public-official.json: {public_files}')
data_root = public_files[0].parent
train_file = data_root / 'train.json'
if not train_file.is_file():
    raise FileNotFoundError(f'Thiếu official train.json cạnh public: {train_file}')
corpus_dirs = [data_root / 'selected-contexts'] if (data_root / 'selected-contexts').is_dir() else []
if not corpus_dirs:
    context_files = list(data_root.rglob('context_*.json'))
    parent_counts = {}
    for path in context_files:
        parent_counts[path.parent] = parent_counts.get(path.parent, 0) + 1
    corpus_dirs = [max(parent_counts, key=parent_counts.get)] if parent_counts else []
if len(corpus_dirs) != 1:
    raise FileNotFoundError(f'Không xác định được corpus: {corpus_dirs}')
EXPECTED_SOURCE_CONTRACT = {
    'schema': 'legalqa-source-contract-v1',
    'release': '0.9.2-v5007',
    'agentic_namespace': 'agentic-v3',
    'cache_schema': 'inference-v7-agentic-error-isolation',
    'selector_schema': 'selector-weights-v3-agentic-evidence',
    'adapter_alias_policy': 'sha256-dedupe-v1',
    'source_alias_policy': 'sha256-dedupe-v1',
    'qlora_recovery_schema': 'legalqa-qlora-recovery-v1',
    'inference_pause_schema': 'inference-runtime-pause-v1',
    'offline_wheel_profile': 'hf-stack-2026-08-v1',
    'offline_wheel_count': 12,
}
REQUIRED_WHEELS = ('transformers-5.14.1-', 'tokenizers-0.22.2-', 'huggingface_hub-1.26.1-', 'hf_xet-1.6.0-', 'safetensors-0.8.0-', 'accelerate-1.14.0-', 'peft-0.18.1-', 'bitsandbytes-0.49.2-', 'sentence_transformers-5.7.0-', 'sentencepiece-0.2.1-', 'regex-2026.7.19-', 'click-8.4.2-')
def source_contract_errors(path):
    errors = []
    marker = path / 'SOURCE_CONTRACT.json'
    try:
        contract = json.loads(marker.read_text(encoding='utf-8'))
    except Exception as exc:
        return [f'thiếu/hỏng SOURCE_CONTRACT.json: {exc}']
    for key, expected in EXPECTED_SOURCE_CONTRACT.items():
        if contract.get(key) != expected:
            errors.append(f'{key}={contract.get(key)!r}, cần {expected!r}')
    required_files = (path / 'kaggle' / 'run_selector_calibration.py', path / 'kaggle' / 'run_inference_only.py', path / 'src' / 'legalqa' / 'selector_schema.py', path / 'RELEASE_V9_2.md', path / 'BASELINE_V5007.md', path / 'configs' / 'selector_v5007.yaml')
    errors.extend(f'thiếu {file.relative_to(path)}' for file in required_files if not file.is_file())
    wheels = list((path / 'wheelhouse').glob('*.whl'))
    names = [wheel.name for wheel in wheels]
    missing_wheels = [prefix for prefix in REQUIRED_WHEELS if not any(name.startswith(prefix) for name in names)]
    if len(wheels) != 12 or missing_wheels or any(wheel.stat().st_size < 10_000 for wheel in wheels):
        errors.append(f'wheelhouse không hợp lệ: count={len(wheels)}, missing={missing_wheels}')
    return errors
def sha256_path(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()
def source_identity(path):
    digest = hashlib.sha256()
    roots = ('SOURCE_CONTRACT.json', 'pyproject.toml', 'RELEASE_V9_2.md', 'configs', 'kaggle', 'scripts', 'src', 'tests', 'wheelhouse')
    files = []
    for name in roots:
        item = path / name
        files.extend([item] if item.is_file() else ([p for p in item.rglob('*') if p.is_file() and '__pycache__' not in p.parts and '.ipynb_checkpoints' not in p.parts and p.suffix != '.pyc'] if item.is_dir() else []))
    for file in sorted(files, key=lambda p: p.relative_to(path).as_posix()):
        digest.update(file.relative_to(path).as_posix().encode('utf-8')); digest.update(b'\0')
        with file.open('rb') as handle:
            for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
                digest.update(block)
    return digest.hexdigest()
source_projects = sorted({p.parent for p in input_root.rglob('pyproject.toml')})
source_reports = {str(path): source_contract_errors(path) for path in source_projects}
qualified_sources = [path for path in source_projects if not source_reports[str(path)]]
source_groups = {}
for path in qualified_sources:
    source_groups.setdefault(source_identity(path), []).append(path)
if len(source_groups) != 1:
    raise RuntimeError(f'Cần đúng một source v9.2 reviewed; source cũ nhúng trong adapter sẽ bị loại. qualified={qualified_sources}; reports={source_reports}')
source_aliases = next(iter(source_groups.values()))
selected_source = min(source_aliases, key=lambda p: (len(p.parts), p.as_posix()))
if len(source_aliases) > 1: print('Source aliases share one SHA-256 identity:', source_aliases)
print('Source v9.2 reviewed:', selected_source)
print('Rejected source copies:', {path: reasons for path, reasons in source_reports.items() if reasons})
source_dir = Path('/kaggle/working/legalqa_source')
if source_dir.exists():
    shutil.rmtree(source_dir)
shutil.copytree(selected_source, source_dir)
project = source_dir
wheels = sorted(str(path) for path in (project / 'wheelhouse').glob('*.whl'))
required = ('transformers-5.14.1-', 'huggingface_hub-1.26.1-', 'hf_xet-1.6.0-', 'click-8.4.2-', 'regex-2026.7.19-', 'tokenizers-0.22.2-', 'accelerate-1.14.0-', 'safetensors-0.8.0-', 'sentence_transformers-5.7.0-', 'peft-0.18.1-', 'bitsandbytes-0.49.2-', 'sentencepiece-0.2.1-')
missing = [prefix for prefix in required if not any(Path(wheel).name.startswith(prefix) for wheel in wheels)]
if missing:
    raise FileNotFoundError(f'Wheelhouse offline chưa đầy đủ: {missing}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '--upgrade', *wheels], check=True)
src_path = str(project / 'src')
sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = src_path + os.pathsep + os.environ.get('PYTHONPATH', '')
os.environ['PYTHONUTF8'] = '1'
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('Môi trường source/offline wheels đã sẵn sàng.')
print('Public:', public_files[0])
print('Train:', train_file)
print('Corpus:', corpus_dirs[0])
print('Model configs:', len(list(input_root.rglob('config.json'))))
print('Source:', selected_source)

In [ ]:
from legalqa.fingerprint import adapter_fingerprint
import yaml

EXPECTED_SELECTOR_SCHEMA = 'selector-weights-v3-agentic-evidence'

def read_json(path, label):
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
    except Exception as exc:
        raise RuntimeError(f'{label} không đọc được: {path}: {exc}') from exc
    if not isinstance(payload, dict):
        raise RuntimeError(f'{label} phải là JSON Object: {path}')
    return payload

# Only the original 215,267-row dense index is valid for this A/B.
dense_matches = []
for manifest_path in sorted(input_root.rglob('manifest.json')):
    try:
        manifest = read_json(manifest_path, 'Dense manifest')
    except RuntimeError:
        continue
    if (
        manifest.get('schema') == 'global-dense-v1'
        and manifest.get('status') == 'complete'
        and manifest.get('identity') == EXPECTED_DENSE_IDENTITY
    ):
        dense_matches.append(manifest_path.parent)
other_dense = []
for manifest_path in sorted(input_root.rglob('manifest.json')):
    try:
        manifest = read_json(manifest_path, 'Dense manifest')
    except RuntimeError:
        continue
    if manifest.get('schema') == 'global-dense-v1' and manifest.get('status') == 'complete' and manifest.get('identity') != EXPECTED_DENSE_IDENTITY:
        other_dense.append(manifest_path.parent)
if other_dense:
    raise RuntimeError(f'Không attach dense index khác trong A1: {other_dense}')
if len(dense_matches) != 1:
    raise RuntimeError(f'A1 cần đúng một dense v0.5007 identity={EXPECTED_DENSE_IDENTITY}: {dense_matches}')
dense_index_input = dense_matches[0]

# The clean adapter must be the only full adapter and must match its content fingerprint.
all_full_adapters = []
for marker in sorted(input_root.rglob('training_complete.json')):
    adapter = marker.parent
    if not (adapter / 'adapter_config.json').is_file():
        continue
    scope_path = adapter / 'qlora_scope.json'
    if not scope_path.is_file():
        continue
    if read_json(scope_path, 'QLoRA scope').get('train_scope') == 'full':
        all_full_adapters.append(adapter)
adapter_identities = {str(path): adapter_fingerprint(path) for path in all_full_adapters}
unexpected_adapters = {path: identity for path, identity in adapter_identities.items() if identity != EXPECTED_ADAPTER_FINGERPRINT}
if unexpected_adapters:
    raise RuntimeError(f'Có adapter full khác A1 được attach: {unexpected_adapters}')
adapter_paths = [Path(path) for path, identity in adapter_identities.items() if identity == EXPECTED_ADAPTER_FINGERPRINT]
if len(adapter_paths) != 1:
    raise RuntimeError(f'A1 cần đúng một adapter clean full fingerprint={EXPECTED_ADAPTER_FINGERPRINT}: {adapter_identities}')
generator_adapter = adapter_paths[0]

# Use the exact v0.5007 selector artifact, never a v4 selector or embedded approximation.
weight_candidates = []
unexpected_weights = {}
for path in sorted(input_root.rglob('tuned_selection.yaml')):
    digest = sha256_path(path)
    if digest == EXPECTED_SELECTOR_SHA256:
        payload = yaml.safe_load(path.read_text(encoding='utf-8'))
        if isinstance(payload, dict) and payload.get('schema') == EXPECTED_SELECTOR_SCHEMA:
            weight_candidates.append(path)
    else:
        unexpected_weights[str(path)] = digest
if unexpected_weights:
    raise RuntimeError(f'Không attach tuned_selection.yaml khác trong A1: {unexpected_weights}')
if len(weight_candidates) != 1:
    raise RuntimeError(f'A1 cần đúng selector v0.5007 SHA={EXPECTED_SELECTOR_SHA256}: {weight_candidates}')
selection_weights = weight_candidates[0]

def recovery_state(path):
    if path.is_file():
        with zipfile.ZipFile(path) as archive:
            state = json.loads(archive.read('recovery_state.json').decode('utf-8'))
    else:
        state = read_json(path / 'recovery_state.json', 'Recovery state')
    if not isinstance(state, dict):
        raise RuntimeError(f'Recovery state không hợp lệ: {path}')
    return state

recovery_candidates = []
for path in sorted(input_root.rglob('legalqa_recovery_latest.zip')):
    if recovery_state(path).get('run_fingerprint') == EXPECTED_RUN_FINGERPRINT:
        recovery_candidates.append(path)
for state_path in sorted(input_root.rglob('recovery_state.json')):
    root = state_path.parent
    if root.name.lower().replace('-', '_') == 'legalqa_recovery_latest' and recovery_state(root).get('run_fingerprint') == EXPECTED_RUN_FINGERPRINT:
        recovery_candidates.append(root)

if AUTO_RESUME:
    if len(recovery_candidates) != 1:
        raise RuntimeError(f'Resume A1 cần đúng một recovery fingerprint={EXPECTED_RUN_FINGERPRINT}: {recovery_candidates}')
    resume_bundle = recovery_candidates[0]
else:
    any_inference_recovery = list(input_root.rglob('legalqa_recovery_latest.zip')) + [
        path.parent for path in input_root.rglob('recovery_state.json')
        if path.parent.name.lower().replace('-', '_') == 'legalqa_recovery_latest'
    ]
    if any_inference_recovery:
        raise RuntimeError(f'Phiên A1 đầu phải không attach inference recovery: {any_inference_recovery}')
    resume_bundle = None

preflight_output = Path('/kaggle/working/v5007_adapter_ablation_preflight.json')
preflight_command = [
    sys.executable, str(project / 'scripts' / 'verify_v5007_adapter_ablation.py'),
    '--public-file', str(public_files[0]),
    '--train-file', str(train_file),
    '--corpus-dir', str(corpus_dirs[0]),
    '--dense-index-dir', str(dense_index_input),
    '--adapter-dir', str(generator_adapter),
    '--selector-weights', str(selection_weights),
    '--output', str(preflight_output),
]
subprocess.run(preflight_command, check=True)
print('A1 adapter:', generator_adapter)
print('A1 dense:', dense_index_input)
print('A1 selector:', selection_weights)
print('A1 recovery:', resume_bundle or 'none (fresh run)')

command = [
    sys.executable, str(project / 'kaggle' / 'run_inference_only.py'),
    '--config', str(project / 'configs' / 'kaggle_inference_t4.yaml'),
    '--input-root', '/kaggle/input',
    '--working', '/kaggle/working/legalqa_inference',
    '--submission-output', '/kaggle/working/submission.zip',
    '--recovery-output', RECOVERY_OUTPUT,
    '--checkpoint-every', str(CHECKPOINT_EVERY),
    '--max-runtime-hours', str(MAX_RUNTIME_HOURS),
    '--device', DEVICE,
    '--embedding-batch-size', str(EMBEDDING_BATCH_SIZE),
    '--reranker-batch-size', str(RERANKER_BATCH_SIZE),
    '--verifier-batch-size', str(VERIFIER_BATCH_SIZE),
    '--reranker-max-length', str(RERANKER_MAX_LENGTH),
    '--max-context-chars', str(MAX_CONTEXT_CHARS),
    '--max-new-tokens', str(MAX_NEW_TOKENS),
    '--bm25-top-k', str(BM25_TOP_K),
    '--dense-top-k', str(DENSE_TOP_K),
    '--fused-top-k', str(FUSED_TOP_K),
    '--rerank-top-k', str(RERANK_TOP_K),
    '--evidence-top-k', str(EVIDENCE_TOP_K),
    '--dense-index-dir', DENSE_INDEX_DIR,
    '--dense-index-input', str(dense_index_input),
    '--generator-adapter', str(generator_adapter),
    '--selection-weights', str(selection_weights),
    '--expected-generator-adapter-fingerprint', EXPECTED_ADAPTER_FINGERPRINT,
    '--expected-run-fingerprint', EXPECTED_RUN_FINGERPRINT,
    '--require-generator-adapter',
    '--require-selection-weights',
    '--require-prebuilt-indexes',
    '--no-train-memory',
    '--reuse-retrieval' if REUSE_RETRIEVAL else '--no-reuse-retrieval',
    '--model-smoke' if MODEL_SMOKE else '--no-model-smoke',
    '--auto-resume' if AUTO_RESUME else '--no-auto-resume',
]
if resume_bundle is not None:
    command.extend(['--resume-bundle', str(resume_bundle)])
if DENSE_INDEX_ONLY:
    raise RuntimeError('A1 is inference-only; DENSE_INDEX_ONLY must stay False.')
if FORCE_DENSE_INDEX or FORCE_INDEX:
    raise RuntimeError('A1 must not rebuild or force an index.')
print(' '.join(command))
subprocess.run(command, check=True)

In [ ]:
import json, zipfile

preflight = json.loads(Path('/kaggle/working/v5007_adapter_ablation_preflight.json').read_text(encoding='utf-8'))
assert preflight['status'] == 'ok'
assert preflight['expected_run_fingerprint'] == EXPECTED_RUN_FINGERPRINT

submission = Path('/kaggle/working/submission.zip')
recovery = Path(RECOVERY_OUTPUT)
manifest_path = Path('/kaggle/working/legalqa_inference/run_manifest.json')
if not manifest_path.is_file():
    raise RuntimeError(f'Thiếu run manifest: {manifest_path}')
run_manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
if run_manifest.get('run_fingerprint') != EXPECTED_RUN_FINGERPRINT:
    raise RuntimeError(f'Fingerprint sai: {run_manifest.get("run_fingerprint")} != {EXPECTED_RUN_FINGERPRINT}')
if not recovery.is_file():
    raise RuntimeError('Thiếu recovery ZIP sau A1 inference.')
with zipfile.ZipFile(recovery) as archive:
    state = json.loads(archive.read('recovery_state.json').decode('utf-8'))
if state.get('run_fingerprint') != EXPECTED_RUN_FINGERPRINT:
    raise RuntimeError(f'Recovery fingerprint sai: {state}')
print('RECOVERY:', recovery, state)

status = run_manifest.get('status')
if status == 'paused':
    if submission.exists():
        raise RuntimeError('Run paused nhưng vẫn có submission cũ; không được nộp.')
    print('PAUSED SAFELY. Save Version, tạo Dataset từ recovery ZIP này, attach lại đúng input A1, rồi đặt AUTO_RESUME=True.')
elif status == 'complete':
    if not submission.is_file():
        raise RuntimeError('Manifest complete nhưng thiếu submission.zip.')
    with zipfile.ZipFile(submission) as archive:
        if archive.namelist() != ['submission.json']:
            raise RuntimeError(f'ZIP chứa sai file: {archive.namelist()}')
        payload = json.loads(archive.read('submission.json').decode('utf-8'))
    official_payload = json.loads(public_files[0].read_text(encoding='utf-8'))
    if set(payload) != set(official_payload):
        raise RuntimeError('question_id submission không khớp public-official.json.')
    if not all(isinstance(item, dict) and isinstance(item.get('answer'), str) and item['answer'].strip() for item in payload.values()):
        raise RuntimeError('Có answer rỗng hoặc không phải string.')
    print(f'A1 READY: {submission} — {submission.stat().st_size:,} bytes — {len(payload)} answers')
else:
    raise RuntimeError(f'Run status không hợp lệ: {status!r}')